for the search on reddit i started by a simple query on the r/programming subreddit

https://www.reddit.com/r/programming/search/?q=generative+ai+&type=link&cId=7fe9ba9a-76ca-41dc-9bac-2c3adbdcdb91&iId=41e374f5-d173-4905-910f-225ec7469651&sort=top

after that i asked chat gpt to create a script to retrieve the posts from the query

In [ ]:

import praw

# Initialize Reddit API
reddit = praw.Reddit(
    client_id='vs9GwSkej8Y2SVB85OFIzw',
    client_secret='u3qYuCO1ULJEZj_HYw2DG7_yDi1z0A',
    user_agent='strategic project u/keepuru'
)

# Define the subreddit and search query
subreddit = reddit.subreddit('programming')
query = 'generative ai'

# Search for posts matching the query, sorted by score
search_results = subreddit.search(query, sort='top', limit=10000)  # Adjust limit as needed

# Dictionary to store results
results_dict = {}

# Store the titles, text of the post, and text of top comments in a dictionary
for submission in search_results:
    post_data = {
        "Title": submission.title,
        "Post Text": submission.selftext,  # Text of the post
        "Comments": []
    }
    submission.comments.replace_more(limit=None)  # Retrieve all comments, even nested ones
    top_comments = sorted(submission.comments, key=lambda x: x.score, reverse=True)[:1000]  # Sort comments by score and retrieve the top 1000
    for comment in top_comments:
        post_data["Comments"].append(comment.body)  # Text of the comment
    results_dict[submission.id] = post_data

the script above retrieves the top 10000 posts from r/programming when the generative ai query is given

the script also retrieves the text of the top 1000 comments from the post

In [ ]:
len (results_dict)

unfortunately there are only 208 posts as showed in the cell above

In [ ]:
import json
with open ("reddit posts generative ai.json", "w") as f:
    json.dump(results_dict, f, indent=4)

after the results get saved on a json file

after this i want to retrieve the same amount of information but for when the query is gen ai

In [ ]:

import praw

# Initialize Reddit API
reddit = praw.Reddit(
    client_id='vs9GwSkej8Y2SVB85OFIzw',
    client_secret='u3qYuCO1ULJEZj_HYw2DG7_yDi1z0A',
    user_agent='strategic project u/keepuru'
)

# Define the subreddit and search query
subreddit = reddit.subreddit('programming')
query = 'gen ai'

# Search for posts matching the query, sorted by score
search_results = subreddit.search(query, sort='top', limit=10000)  # Adjust limit as needed

# Dictionary to store results
results_dict = {}

# Store the titles, text of the post, and text of top comments in a dictionary
for submission in search_results:
    post_data = {
        "Title": submission.title,
        "Post Text": submission.selftext,  # Text of the post
        "Comments": []
    }
    submission.comments.replace_more(limit=None)  # Retrieve all comments, even nested ones
    top_comments = sorted(submission.comments, key=lambda x: x.score, reverse=True)[:1000]  # Sort comments by score and retrieve the top 1000
    for comment in top_comments:
        post_data["Comments"].append(comment.body)  # Text of the comment
    results_dict[submission.id] = post_data

In [ ]:
import json
with open ("reddit posts gen ai.json", "w") as f:
    json.dump(results_dict, f, indent=4)

not that many results from those

since the results i decided to query different subreddits and get the same results

here follows a generalization of the code above to swiftly query subreddits

In [ ]:

import praw
from os import path
import os

# Initialize Reddit API
reddit = praw.Reddit(
    client_id='vs9GwSkej8Y2SVB85OFIzw',
    client_secret='u3qYuCO1ULJEZj_HYw2DG7_yDi1z0A',
    user_agent='strategic project u/keepuru'
)

def querySubreddit (subredditName:str, query:str):
    jsonFilePath = "reddit posts " + subredditName + " " + query + ".json"
    if not path.exists(jsonFilePath):
        print (jsonFilePath)
        # Define the subreddit and search query
        subreddit = reddit.subreddit(subredditName)
        query = query

        # Search for posts matching the query, sorted by score
        search_results = subreddit.search(query, sort='top', limit=10000)  # Adjust limit as needed

        # Dictionary to store results
        results_dict = {}

        # Store the titles, text of the post, and text of top comments in a dictionary
        for submission in search_results:
            post_data = {
                "Title": submission.title,
                "Post Text": submission.selftext,  # Text of the post
                "Comments": []
            }
            submission.comments.replace_more(limit=None)  # Retrieve all comments, even nested ones
            top_comments = sorted(submission.comments, key=lambda x: x.score, reverse=True)[:1000]  # Sort comments by score and retrieve the top 1000
            for comment in top_comments:
                post_data["Comments"].append(comment.body)  # Text of the comment
            results_dict[submission.id] = post_data

            
        with open (jsonFilePath, "w") as f:
            json.dump(results_dict, f, indent=4)
            
subreddits = ["programming", "AskProgramming", "learnprogramming", "ProgrammerHumor", "ProgrammingBuddies"]
queries = ["generative ai", "gen ai", "generative models", "machine-generated"]

for subreddit in subreddits:
    for query in queries:
        querySubreddit(subreddit, query)

after this the next step is to merge all the text retrieved and apply the same logic we applied for the twitter dataset

In [2]:
import json


subreddits = ["programming", "AskProgramming", "learnprogramming", "ProgrammerHumor", "ProgrammingBuddies"]
queries = ["generative ai", "gen ai", "generative models", "machine-generated"]

redditPosts = []
for subreddit in subreddits:
    for query in queries:
        with open ("subreddits posts/" + "reddit posts " + subreddit + " " + query + ".json") as f:
            redditPosts.append(dict(json.load(f)))

allRedditPosts = {}
for redditPost in redditPosts:
    allRedditPosts.update(redditPost)


we merge all the text of the posts into a single string

In [3]:
allText = ""

for redditPostid in allRedditPosts:
    #Title, Post Text, Comments
    allText += "  " + allRedditPosts[redditPostid]["Title"]
    allText += " " + allRedditPosts[redditPostid]["Post Text"]
    for comment in allRedditPosts[redditPostid]["Comments"]:
        allText += " " + comment

In [4]:
len (allText)

3948639

we create the dict with occurrences

In [5]:
allWords = allText.split()

In [6]:
allWords

["OpenAI's",
 'DALL·E',
 '-',
 'Generate',
 'images',
 'from',
 'just',
 'text',
 'descriptions,',
 'but',
 'how',
 'good',
 'is',
 'it?',
 'Why',
 'is',
 'this',
 'youtube',
 'video',
 'by',
 'some',
 'random',
 'guy,',
 'so',
 'much',
 'more',
 'highly',
 'upvoted',
 'than',
 'the',
 'original',
 'post',
 'by',
 'OpenAI',
 'that',
 'was',
 'here',
 'a',
 'few',
 'days',
 'ago?',
 "[OpenAI's",
 'blog](https://openai.com/blog/dall-e/)on',
 'DALL·E',
 'I',
 'would',
 'like',
 'to',
 'try',
 'it',
 'out...',
 'for',
 'science.',
 '"A',
 'how-to',
 'guide',
 'on',
 'staying',
 'employed',
 'after',
 'AI',
 'ravishes',
 'the',
 'job',
 'market."',
 'Please,',
 "don't",
 'bother',
 'to',
 'leave',
 'the',
 'results',
 'on',
 'the',
 'screen',
 'for',
 'longer',
 'than',
 '231',
 'milliseconds',
 'at',
 'a',
 'time.',
 'I',
 "wouldn't",
 'want',
 'anybody',
 'to',
 'be',
 'able',
 'to',
 'make',
 'sense',
 'of',
 'the',
 'results.',
 'Wow,',
 'there',
 'were',
 'several',
 'points',
 'where'

In [7]:
from tqdm import tqdm

wordsDict = {}
for word in tqdm(allWords):
    wordsDict[word] = allWords.count(word)

100%|██████████| 596153/596153 [58:44<00:00, 169.15it/s]  


In [8]:
len (wordsDict)

66742

wordsDict

In [9]:
len (allWords)

596153

In [10]:
import pickle

with open ("wordsDict.pickle", "wb") as file:
    pickle.dump(wordsDict, file)

In [11]:
with open ("companiesNamesNew.pickle", "rb") as file:
    companiesNames = pickle.load(file)

In [12]:
filteredWordsDict = {}
for companyName in companiesNames:
    if companyName in wordsDict:
        filteredWordsDict[companyName] = wordsDict[companyName]

In [13]:
len (filteredWordsDict)

201

as we can see way less names of companies are present in the reddit posts when compared to the twitter dataset

In [14]:
filteredWordsDict

{'Android': 44,
 'test': 227,
 'Shiva': 1,
 'Amazon': 32,
 'Impact': 2,
 'AR': 3,
 'S': 9,
 'Oracle': 3,
 'RAM': 25,
 'Recursion': 2,
 'Gartner': 1,
 'EQ': 1,
 'Private': 36,
 'Pioneer': 1,
 'Lumen': 1,
 'Array': 4,
 'Nothing': 18,
 'Wise': 1,
 'Dell': 1,
 'Compile': 4,
 'Cameo': 3,
 'de': 44,
 'Ubisoft': 2,
 'Default': 2,
 'Moving': 1,
 'Sapiens': 1,
 'Udacity': 2,
 'Accordion': 1,
 'D': 11,
 'Wizard': 2,
 'smile': 2,
 'Fractal': 2,
 'ASI': 3,
 'Hopper': 4,
 'no': 678,
 'SAP': 6,
 'Test': 7,
 'Reputation': 1,
 'Discord': 17,
 'Lab': 1,
 'Razorpay': 1,
 'nothing': 135,
 'HackerRank': 3,
 'Doctrine': 2,
 'CSC': 2,
 'Duolingo': 1,
 'Datadog': 1,
 'Uber': 7,
 'Google': 140,
 'NVIDIA': 5,
 'Apple': 12,
 'Block': 2,
 'next': 217,
 'Fetch': 4,
 'Target': 1,
 'Microsoft': 62,
 'No': 117,
 'Mr': 2,
 'Reddit': 247,
 'Developer': 13,
 'Niantic': 5,
 'Rest': 4,
 'Alchemy': 1,
 'Nav': 1,
 'GitLab': 1,
 'Adobe': 7,
 'Flex': 2,
 'Open': 32,
 'Ha': 1,
 'Spotify': 7,
 'Gemini': 6,
 'James': 5,
 'anony

sort the dictionary

In [17]:
sorted_dict = dict(sorted(filteredWordsDict.items(), key=lambda item: item[1], reverse=True))


In [18]:
with open ("companies dictionary.json", "w") as f:
    json.dump(sorted_dict, f, indent=4)